# 51. The Classic Economic Order Quantity (EOQ) Problem

## Tier 2: The Classic Heuristic (Python Implementation)

### Key Assumptions
- Same fundamental assumptions as Tier 1 (deterministic demand, constant costs)
- Computational approach enables practical implementation
- Enhanced with practical business considerations
- Includes constraint handling and sensitivity analysis

### Algorithm: EOQ Classic Heuristic

The classic EOQ heuristic provides a straightforward computational approach that:
1. **Implements the standard EOQ formula** with robust error handling
2. **Adds practical business logic** for real-world implementation
3. **Performs comprehensive sensitivity analysis** for parameter variations
4. **Generates actionable insights** including order schedules and cost breakdowns

### Time Complexity: O(1) for basic EOQ calculation, O(n) for sensitivity analysis
### Space Complexity: O(1) for core calculation, O(n) for storing sensitivity results

### What to Look For in Results
- Immediate EOQ calculation with business-friendly output
- Comprehensive cost breakdown and analysis
- Sensitivity analysis showing parameter impacts
- Practical order schedule for implementation
- Comparison with different parameter scenarios

In [1]:
# Import required libraries for heuristic implementation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set professional styling for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
class EOQClassicHeuristic:
    """
    Classic EOQ heuristic implementation with practical business considerations.
    This implementation goes beyond basic mathematical formula to include
    practical features for real-world inventory management.
    """
    
    def __init__(self, demand_rate, ordering_cost, holding_cost, unit_cost=None):
        """
        Initialize EOQ heuristic with problem parameters.
        
        Parameters:
        - demand_rate: Annual demand in units per year
        - ordering_cost: Fixed cost per order in dollars
        - holding_cost: Annual holding cost per unit in dollars
        - unit_cost: Cost per unit (optional, for total cost calculation)
        """
        self.D = float(demand_rate)
        self.S = float(ordering_cost)
        self.H = float(holding_cost)
        self.c = float(unit_cost) if unit_cost is not None else None
        
        # Validate inputs
        self._validate_inputs()
        
    def _validate_inputs(self):
        """
        Validate input parameters for business logic.
        """
        if self.D <= 0:
            raise ValueError("Demand rate must be positive")
        if self.S <= 0:
            raise ValueError("Ordering cost must be positive")
        if self.H <= 0:
            raise ValueError("Holding cost must be positive")
        if self.c is not None and self.c <= 0:
            raise ValueError("Unit cost must be positive")
    
    def calculate_eoq(self):
        """
        Calculate Economic Order Quantity using the classic formula.
        
        Formula: EOQ = sqrt(2 * D * S / H)
        """
        eoq = np.sqrt(2 * self.D * self.S / self.H)
        return eoq
    
    def calculate_comprehensive_metrics(self, eoq=None):
        """
        Calculate comprehensive set of business metrics.
        """
        if eoq is None:
            eoq = self.calculate_eoq()
        
        # Basic EOQ metrics
        orders_per_year = self.D / eoq
        days_between_orders = 365 / orders_per_year
        avg_inventory = eoq / 2
        
        # Cost calculations
        annual_ordering_cost = orders_per_year * self.S
        annual_holding_cost = avg_inventory * self.H
        annual_inventory_cost = annual_ordering_cost + annual_holding_cost
        
        # Total cost including purchasing
        total_annual_cost = annual_inventory_cost
        annual_purchasing_cost = 0
        if self.c is not None:
            annual_purchasing_cost = self.D * self.c
            total_annual_cost += annual_purchasing_cost
        
        # Additional business metrics
        inventory_turnover = self.D / avg_inventory
        days_of_supply = eoq / (self.D / 365)
        
        return {
            'eoq': eoq,
            'orders_per_year': orders_per_year,
            'days_between_orders': days_between_orders,
            'avg_inventory': avg_inventory,
            'annual_ordering_cost': annual_ordering_cost,
            'annual_holding_cost': annual_holding_cost,
            'annual_inventory_cost': annual_inventory_cost,
            'annual_purchasing_cost': annual_purchasing_cost,
            'total_annual_cost': total_annual_cost,
            'inventory_turnover': inventory_turnover,
            'days_of_supply': days_of_supply
        }
    
    def sensitivity_analysis(self, variation_pct=0.2, steps=5):
        """
        Perform comprehensive sensitivity analysis.
        
        Parameters:
        - variation_pct: Percentage variation for each parameter (default 20%)
        - steps: Number of variation steps (default 5)
        """
        base_eoq = self.calculate_eoq()
        base_metrics = self.calculate_comprehensive_metrics(base_eoq)
        
        # Generate variation steps
        variations = np.linspace(1 - variation_pct, 1 + variation_pct, steps)
        
        results = {
            'variation_multipliers': variations.tolist(),
            'demand_impact': [],
            'ordering_cost_impact': [],
            'holding_cost_impact': [],
            'unit_cost_impact': [] if self.c is not None else None
        }
        
        # Test demand variations
        for var in variations:
            temp_model = EOQClassicHeuristic(self.D * var, self.S, self.H, self.c)
            eoq_var = temp_model.calculate_eoq()
            metrics_var = temp_model.calculate_comprehensive_metrics(eoq_var)
            results['demand_impact'].append({
                'multiplier': var,
                'eoq': eoq_var,
                'total_cost': metrics_var['total_annual_cost'],
                'cost_change_pct': ((metrics_var['total_annual_cost'] - base_metrics['total_annual_cost']) / base_metrics['total_annual_cost']) * 100
            })
        
        # Test ordering cost variations
        for var in variations:
            temp_model = EOQClassicHeuristic(self.D, self.S * var, self.H, self.c)
            eoq_var = temp_model.calculate_eoq()
            metrics_var = temp_model.calculate_comprehensive_metrics(eoq_var)
            results['ordering_cost_impact'].append({
                'multiplier': var,
                'eoq': eoq_var,
                'total_cost': metrics_var['total_annual_cost'],
                'cost_change_pct': ((metrics_var['total_annual_cost'] - base_metrics['total_annual_cost']) / base_metrics['total_annual_cost']) * 100
            })
        
        # Test holding cost variations
        for var in variations:
            temp_model = EOQClassicHeuristic(self.D, self.S, self.H * var, self.c)
            eoq_var = temp_model.calculate_eoq()
            metrics_var = temp_model.calculate_comprehensive_metrics(eoq_var)
            results['holding_cost_impact'].append({
                'multiplier': var,
                'eoq': eoq_var,
                'total_cost': metrics_var['total_annual_cost'],
                'cost_change_pct': ((metrics_var['total_annual_cost'] - base_metrics['total_annual_cost']) / base_metrics['total_annual_cost']) * 100
            })
        
        # Test unit cost variations (if applicable)
        if self.c is not None:
            for var in variations:
                temp_model = EOQClassicHeuristic(self.D, self.S, self.H, self.c * var)
                eoq_var = temp_model.calculate_eoq()
                metrics_var = temp_model.calculate_comprehensive_metrics(eoq_var)
                results['unit_cost_impact'].append({
                    'multiplier': var,
                    'eoq': eoq_var,
                    'total_cost': metrics_var['total_annual_cost'],
                    'cost_change_pct': ((metrics_var['total_annual_cost'] - base_metrics['total_annual_cost']) / base_metrics['total_annual_cost']) * 100
                })
        
        return results
    
    def generate_order_schedule(self, start_date=None, periods=12):
        """
        Generate practical order schedule for implementation.
        
        Parameters:
        - start_date: Starting date for schedule (default: today)
        - periods: Number of months to generate schedule for
        """
        if start_date is None:
            start_date = datetime.now().date()
        
        metrics = self.calculate_comprehensive_metrics()
        days_between_orders = metrics['days_between_orders']
        eoq = metrics['eoq']
        
        schedule = []
        current_date = start_date
        end_date = start_date + timedelta(days=periods * 30)
        order_num = 1
        
        while current_date <= end_date:
            schedule.append({
                'order_number': order_num,
                'order_date': current_date,
                'order_quantity': eoq,
                'expected_delivery': current_date + timedelta(days=1),  # Assuming 1-day lead time
                'notes': f'Order #{order_num} - {eoq:.1f} units'
            })
            current_date += timedelta(days=days_between_orders)
            order_num += 1
        
        return schedule
    
    def compare_with_alternatives(self, alternative_eoqs=None):
        """
        Compare optimal EOQ with alternative order quantities.
        """
        if alternative_eoqs is None:
            optimal_eoq = self.calculate_eoq()
            alternative_eoqs = [
                optimal_eoq * 0.5,   # Half of optimal
                optimal_eoq * 0.75,  # 75% of optimal
                optimal_eoq * 1.25,  # 125% of optimal
                optimal_eoq * 1.5,   # 150% of optimal
                optimal_eoq * 2.0    # Double of optimal
            ]
        
        optimal_metrics = self.calculate_comprehensive_metrics()
        comparisons = []
        
        for alt_eoq in alternative_eoqs:
            alt_metrics = self.calculate_comprehensive_metrics(alt_eoq)
            cost_increase = ((alt_metrics['total_annual_cost'] - optimal_metrics['total_annual_cost']) / optimal_metrics['total_annual_cost']) * 100
            
            comparisons.append({
                'order_quantity': alt_eoq,
                'total_cost': alt_metrics['total_annual_cost'],
                'cost_increase_pct': cost_increase,
                'orders_per_year': alt_metrics['orders_per_year'],
                'avg_inventory': alt_metrics['avg_inventory']
            })
        
        return comparisons

In [3]:
# Visualization functions for the heuristic approach

def plot_comprehensive_analysis(model):
    """
    Create a comprehensive 4-panel analysis dashboard.
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Get base metrics
    metrics = model.calculate_comprehensive_metrics()
    optimal_eoq = metrics['eoq']
    
    # Panel 1: Cost Components
    eoq_range = np.linspace(optimal_eoq * 0.3, optimal_eoq * 2.5, 100)
    ordering_costs = []
    holding_costs = []
    total_costs = []
    
    for eoq in eoq_range:
        m = model.calculate_comprehensive_metrics(eoq)
        ordering_costs.append(m['annual_ordering_cost'])
        holding_costs.append(m['annual_holding_cost'])
        total_costs.append(m['annual_inventory_cost'])
    
    ax1.plot(eoq_range, ordering_costs, 'b-', linewidth=2.5, label='Ordering Cost', alpha=0.8)
    ax1.plot(eoq_range, holding_costs, 'r-', linewidth=2.5, label='Holding Cost', alpha=0.8)
    ax1.plot(eoq_range, total_costs, 'g-', linewidth=3, label='Total Cost', alpha=0.9)
    ax1.plot(optimal_eoq, metrics['annual_inventory_cost'], 'ko', markersize=10, label=f'Optimal: {optimal_eoq:.1f}')
    ax1.axvline(x=optimal_eoq, color='black', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Order Quantity (units)', fontweight='bold')
    ax1.set_ylabel('Annual Cost ($)', fontweight='bold')
    ax1.set_title('Cost Components Analysis', fontweight='bold', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # Panel 2: Cost Breakdown Pie Chart
    if model.c is not None:
        costs = [metrics['annual_ordering_cost'], metrics['annual_holding_cost'], metrics['annual_purchasing_cost']]
        labels = ['Ordering Cost', 'Holding Cost', 'Purchasing Cost']
        colors = ['#ff9999', '#66b3ff', '#99ff99']
    else:
        costs = [metrics['annual_ordering_cost'], metrics['annual_holding_cost']]
        labels = ['Ordering Cost', 'Holding Cost']
        colors = ['#ff9999', '#66b3ff']
    
    wedges, texts, autotexts = ax2.pie(costs, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax2.set_title(f'Annual Cost Breakdown\nTotal: ${metrics["total_annual_cost"]:,.0f}', fontweight='bold', fontsize=14)
    
    # Panel 3: Sensitivity Analysis
    sensitivity = model.sensitivity_analysis()
    variations = sensitivity['variation_multipliers']
    
    demand_eoqs = [item['eoq'] for item in sensitivity['demand_impact']]
    order_eoqs = [item['eoq'] for item in sensitivity['ordering_cost_impact']]
    hold_eoqs = [item['eoq'] for item in sensitivity['holding_cost_impact']]
    
    ax3.plot(variations, demand_eoqs, 'bo-', linewidth=2.5, markersize=6, label='Demand')
    ax3.plot(variations, order_eoqs, 'go-', linewidth=2.5, markersize=6, label='Order Cost')
    ax3.plot(variations, hold_eoqs, 'mo-', linewidth=2.5, markersize=6, label='Holding Cost')
    ax3.axhline(y=optimal_eoq, color='red', linestyle='--', alpha=0.7, label='Base EOQ')
    ax3.set_xlabel('Parameter Multiplier', fontweight='bold')
    ax3.set_ylabel('Optimal EOQ (units)', fontweight='bold')
    ax3.set_title('Parameter Sensitivity Analysis', fontweight='bold', fontsize=14)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Alternative Comparisons
    comparisons = model.compare_with_alternatives()
    order_quantities = [comp['order_quantity'] for comp in comparisons]
    cost_increases = [comp['cost_increase_pct'] for comp in comparisons]
    
    bars = ax4.bar(range(len(comparisons)), cost_increases, color=['red' if x > 5 else 'orange' if x > 2 else 'green' for x in cost_increases])
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    ax4.set_xlabel('Alternative Order Quantities', fontweight='bold')
    ax4.set_ylabel('Cost Increase vs Optimal (%)', fontweight='bold')
    ax4.set_title('Alternative Order Quantity Analysis', fontweight='bold', fontsize=14)
    ax4.set_xticks(range(len(comparisons)))
    ax4.set_xticklabels([f'{q:.0f}' for q in order_quantities], rotation=45)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, increase in zip(bars, cost_increases):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{increase:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return fig

### Concrete Example - MegaMart Coffee Beans

Let's apply the classic EOQ heuristic to MegaMart's Colombian Supremo coffee bean inventory problem:

**Problem Parameters:**
- Annual demand (D): 12,000 bags per year
- Ordering cost (S): $150 per order
- Holding cost (H): $8 per bag per year
- Unit cost: $25 per bag (assumed for comprehensive analysis)

In [ ]:
# Initialize the EOQ heuristic with MegaMart's parameters
megamart_heuristic = EOQClassicHeuristic(
    demand_rate=12000,    # 12,000 bags per year
    ordering_cost=150,     # $150 per order
    holding_cost=8,        # $8 per bag per year
    unit_cost=25          # $25 per bag (assumed)
)

# Calculate comprehensive metrics
metrics = megamart_heuristic.calculate_comprehensive_metrics()

print("=== MegaMart Coffee Bean EOQ Analysis ===")
print(f"Optimal Order Quantity: {metrics['eoq']:.2f} bags")
print(f"Orders per Year: {metrics['orders_per_year']:.2f}")
print(f"Days Between Orders: {metrics['days_between_orders']:.1f}")
print(f"Average Inventory Level: {metrics['avg_inventory']:.2f} bags")
print(f"Inventory Turnover: {metrics['inventory_turnover']:.2f} times per year")
print(f"Days of Supply: {metrics['days_of_supply']:.1f} days")
print()
print("=== Cost Breakdown ===")
print(f"Annual Ordering Cost: ${metrics['annual_ordering_cost']:,.2f}")
print(f"Annual Holding Cost: ${metrics['annual_holding_cost']:,.2f}")
print(f"Annual Inventory Cost: ${metrics['annual_inventory_cost']:,.2f}")
print(f"Annual Purchasing Cost: ${metrics['annual_purchasing_cost']:,.2f}")
print(f"Total Annual Cost: ${metrics['total_annual_cost']:,.2f}")

# Verify cost balance at optimal EOQ
cost_balance = abs(metrics['annual_ordering_cost'] - metrics['annual_holding_cost'])
balance_pct = (cost_balance / metrics['annual_inventory_cost']) * 100
print(f"\nCost Balance Verification: |Ordering - Holding| = ${cost_balance:.2f} ({balance_pct:.3f}%)")

In [ ]:
# Generate comprehensive analysis dashboard
fig = plot_comprehensive_analysis(megamart_heuristic)
plt.show()

In [ ]:
# Perform detailed sensitivity analysis
sensitivity_results = megamart_heuristic.sensitivity_analysis(variation_pct=0.2, steps=5)

print("\n=== Detailed Sensitivity Analysis ===")
print("Parameter Variation | EOQ Change | Total Cost Change")
print("-" * 55)

print("\nDemand Variations:")
for item in sensitivity_results['demand_impact']:
    print(f"{item['multiplier']:>17.2f}x | {item['eoq']:>9.1f} | {item['cost_change_pct']:>+11.2f}%")

print("\nOrdering Cost Variations:")
for item in sensitivity_results['ordering_cost_impact']:
    print(f"{item['multiplier']:>17.2f}x | {item['eoq']:>9.1f} | {item['cost_change_pct']:>+11.2f}%")

print("\nHolding Cost Variations:")
for item in sensitivity_results['holding_cost_impact']:
    print(f"{item['multiplier']:>17.2f}x | {item['eoq']:>9.1f} | {item['cost_change_pct']:>+11.2f}%")

if sensitivity_results['unit_cost_impact'] is not None:
    print("\nUnit Cost Variations:")
    for item in sensitivity_results['unit_cost_impact']:
        print(f"{item['multiplier']:>17.2f}x | {item['eoq']:>9.1f} | {item['cost_change_pct']:>+11.2f}%")

In [ ]:
# Generate practical order schedule
schedule = megamart_heuristic.generate_order_schedule(periods=3)  # 3 months

print("\n=== First Quarter Order Schedule ===")
for order in schedule:
    print(f"Day {order['order_date'].strftime('%j')}: Order {order['order_quantity']:.2f} bags")

# Alternative order quantity analysis
comparisons = megamart_heuristic.compare_with_alternatives()

print("\n=== Alternative Order Quantity Analysis ===")
print("Order Qty | Total Cost | Cost Increase | Orders/Year | Avg Inventory")
print("-" * 70)
for comp in comparisons:
    print(f"{comp['order_quantity']:>8.0f} | ${comp['total_cost']:>9.0f} | {comp['cost_increase_pct']:>+11.2f}% | {comp['orders_per_year']:>10.2f} | {comp['avg_inventory']:>11.0f}")

### Why This Heuristic Approach vs Mathematical Formulation

The classic EOQ heuristic provides several advantages over the pure mathematical approach:

#### Advantages:
1. **Practical Implementation**: Ready-to-use functions for business applications
2. **Comprehensive Analysis**: Full cost breakdown including purchasing costs
3. **Sensitivity Insights**: Detailed parameter impact analysis for decision-making
4. **Actionable Outputs**: Order schedules and alternative comparisons
5. **Error Handling**: Robust input validation and business logic
6. **Visualization**: Professional charts and dashboards for stakeholder communication

#### Disadvantages:
1. **Computational Overhead**: Slightly more complex than simple formula
2. **Same Assumptions**: Still inherits deterministic assumptions of base model
3. **No Uncertainty**: Doesn't address demand variability or lead times

#### When to Use This Tier:
- **Business Implementation**: When you need practical tools for daily operations
- **Decision Support**: When presenting to management with comprehensive analysis
- **Parameter Planning**: When testing different scenarios and sensitivity
- **System Integration**: When building inventory management systems

### Key Insights from Heuristic Analysis

The comprehensive heuristic analysis reveals:
1. **Cost Balance**: Optimal EOQ perfectly balances ordering and holding costs
2. **Parameter Sensitivity**: Holding cost changes have the strongest impact on EOQ
3. **Robustness**: Small deviations from optimal EOQ result in minimal cost increases
4. **Practical Scheduling**: Clear order timeline for operational implementation
5. **Business Metrics**: Additional insights like inventory turnover and days of supply

This heuristic approach bridges the gap between theoretical understanding and practical business application, providing the tools needed for real-world inventory management decisions.